# **Bambu HLS tutorial**

## <font color='#088F8F'> **Initial setup** </font>

Bambu is distributed as a standalone **AppImage**, making it easy to run in a **Colab notebook** or on any Linux machine without requiring complex installation steps.


Install Bambu and required packages:

In [ ]:
!echo "deb http://ppa.launchpad.net/git-core/ppa/ubuntu $(cat /etc/os-release | grep UBUNTU_CODENAME | sed 's/.*=//g') main" >> /etc/apt/sources.list.d/git-core.list
!apt-key adv --keyserver keyserver.ubuntu.com --recv-keys A1715D88E1DF1F24
!apt-get update
!apt-get install -y --no-install-recommends build-essential ca-certificates gcc-multilib g++-multilib git libtinfo5 verilator wget
!wget https://release.bambuhls.eu/appimage/bambu-latest.AppImage
!chmod +x bambu-*.AppImage
!ln -sf $PWD/bambu-*.AppImage /bin/bambu
!ln -sf $PWD/bambu-*.AppImage /bin/panda_shell
!ln -sf $PWD/bambu-*.AppImage /bin/spider
!git clone --depth 1 --filter=blob:none --branch dev/panda --sparse https://github.com/ferrandi/PandA-bambu.git
%cd PandA-bambu
!git sparse-checkout set documentation/bambu101
%cd ..
!mv PandA-bambu/documentation/bambu101/* .

To verify that the installation was successful, launch `bambu` through the command-line. You can explore all available options and features by printing its help menu.

In [ ]:
!bambu --help

---

## <font color='#088F8F'> **Exercise 1: inputs and outputs** </font>

The HLS tool requires an input description of the application to be translated into hardware. The input must define a **single top-level function** that handles all inputs and outputs. This function will be synthesized into hardware by Bambu.

In this example, we use a simple CRC kernel implementation as the input application.

You can inspect the content of the source file below:

In [ ]:
!cat /content/basic_usage/icrc.c

### **Minimal set of options**

Let us start by adding a few configuration options that drive the synthesis process.

### Top level interface (mandatory)

Similar to a standard compiler that needs a main function to know where to start, the HLS tool requires that the name of the function to be accelerated is specified through the `--top-fname` command line option.

### Target hardware

We need to specify what is the hardware target for which the RTL description needs to be generated (if unspecified, the default is an AMD Zynq7000 FPGA).
For this example we will select an Alveo U280 platform through the `--device-name=xcu280-2Lfsvh2892-VVD` command line option.   
Furthermore, we specify the required clock period in nanoseconds through the `--clock-period=5` option (the default is 10).

### Verbosity

Controls the amount of information printed on the output (values from 0 to 6, with 5 and 6 used to debug the cosimulation process). We use `-v4` to see all Bambu passes.

### **Synthesis**

Now we are ready to generate the hardware design for the `icrc1` function using the following command:

In [ ]:
%cd /content/basic_usage
!bambu icrc.c --top-fname=icrc1 --device-name=xcu280-2Lfsvh2892-VVD --clock-period=5 -v4

### **Output hardware design**
After the process completes successfully, the generated Verilog design can be found in the `/content/basic_usage/icrc1.v` file.

Additionally, the HLS tool produces a synthesis script named `/content/basic_usage/synthesize_Synthesis_icrc1.sh` which can be used to run the vendor-specific synthesis tool to generate the bitstream for the target FPGA.

We can examine the top-level module and inspect its interface. As no specific interface protocol was specified, we expect to find a `start_port` to trigger the accelerator, along with `done_port`, `clock` and `reset` ports to manage its operation.
Additionally, there should be a `return_port` for the function result and two separate wires corresponding to the two input parameters.

In [ ]:
import re

def print_last_verilog_interface(filename):
    with open(filename, 'r') as f:
        content = f.read()

    modules = re.findall(r'(module\s+\w+\s*\([^;]*?\);)', content, re.DOTALL)

    if not modules:
        print(f"No modules found in {filename}")
        return

    last_module = modules[-1]

    print(f"Interface of the top module in {filename}:\n")
    print(last_module)

print_last_verilog_interface('/content/basic_usage/icrc1.v')

---

## <font color='#088F8F'> **Exercise 2: verification** </font>

Bambu HLS offers the possibility to easily run **verification** of the output design with respect to the input representation. This can be requested by passing a testbench file where the top level function is executed through the `--generate-tb=testbench.c` option.

Inspect the testbench file used to verify the `icrc1` function:

In [ ]:
!cat /content/basic_usage/testbench.c

Launch Bambu requesting to launch cosimulation after Verilog synthesis:

In [ ]:
!bambu icrc.c --top-fname=icrc1 --device-name=xcu280-2Lfsvh2892-VVD --clock-period=5 -v4 --generate-tb=testbench.c --simulate -fno-unroll-loops --print-dot

Inspect the graphical representation of the generated Finite State Machine:

In [ ]:
from graphviz import Source
Source.from_file('HLS_output/dot/icrc1/fsm.dot')

---

## <font color='#088F8F'> **Exercise 3: TrueFloat** </font>

These are the Bambu command-line options that control the generation of **floating-point arithmetic cores**:



```
--fp-exception-mode=<ieee|saturation|overflow>
    Set the soft-based exception handling mode:
        ieee    - IEEE754 standard exceptions (default)
        saturation - Inf is replaced with max value, Nan becomes undefined behaviour
        overflow  - Inf and Nan results in undefined behaviour

--fp-rounding-mode=<nearest_even|truncate>
    Set the soft-based rounding handling mode:
        nearest_even - IEEE754 standard rounding mode (default)
        truncate  - No rounding is applied

--fp-format=<func_name>*e<exp_bits>m<frac_bits>b<exp_bias><rnd_mode><exc_mode><?spec><?sign>
    Define arbitrary precision floating-point format by function (use comma separated
    list for multiple definitions). (i.e.: e8m27b-127nihs represent IEEE754 single precision FP)
        func_name - Set arbitrary floating-point format for a specific function (using
                    @ symbol here will resolve to the top function)
                    (Arbitrary floating-point format will apply to specified function
                    only, use --propagate-fp-format to extend it to called functions)
        exp_bits - Number of bits used by the exponent
        frac_bits - Number of bits used by the fractional value
        exp_bias - Bias applied to the unsigned value represented by the exponent bits
        rnd_mode - Rounding mode (exclusive option):
                        n - nearest_even: IEEE754 standard rounding mode
                        t - truncate    : no rounding is applied
        exc_mode - Exception mode (exclusive option):
                        i - ieee      : IEEE754 standard exceptions
                        a - saturation: Inf is replaced with max value, Nan becomes undefined behaviour
                        o - overflow  : Inf and Nan results in undefined behaviour
            spec   - Floating-point specialization string (multiple choice):
                        h - hidden one: IEEE754 standard representation with hidden one
                        s - subnormals: IEEE754 subnormal numbers
            sign   - Static sign representation (exclusive option):
                        - IEEE754 dynamic sign is used if omitted
                        1 - all values are considered as negative numbers
                        0 - all values are considered as positive numbers

--fp-format=inline-math
    The "inline-math" flag may be added to fp-format option to force floating-point
    arithmetic operators always inline policy

--fp-format=inline-conversion
    The "inline-conversion" flag may be added to fp-format option to force floating-point
    conversion operators always inline policy

--fp-format-interface
    User-defined floating-point format is applied to top interface signature if required
    (default modifies top function body only)

--fp-format-propagate
    Propagate user-defined floating-point format to called function when possible
```

Let's use a trigonometry toy example to show the capabilities of the TrueFloat, with `/content/truefloat/sine_gen.cpp` implementing the testbench and `sine_table_gen` as top module to be synthesized. The application generates a set of points sampling a sine wave with given frequency and amplitude, and it requires five command-line arguments:
```
Usage: sine_gen <samples> <frequency> <amplitude> <sampling rate> <out_file>
```

We need to add a few Bambu options to enable floating-point synthesis and pass command-line argument to the `sine_table_gen` binary:

In [ ]:
%mkdir -p /content/truefloat/baseline
%cd /content/truefloat/baseline
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=baseline.dat --simulate -v1

We can generate variants with different custom precision floating-point formats through the `--fp-format` command line option.

Since we are intentionally losing precision with respect to the original double precision representation, we need to disable the automatic verification against software execution through the `-DCUSTOM_VERIFICATION` command-line option.

As an example, we can generate the following variants:
- e5m5b-15nih

In [ ]:
%mkdir -p /content/truefloat/e5m5b_15nih
%cd /content/truefloat/e5m5b_15nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e5m5b-15nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e5m5b_15nih.dat --simulate -v1

- e4m4b-7nih

In [ ]:
%mkdir -p /content/truefloat/e4m4b_7nih
%cd /content/truefloat/e4m4b_7nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e4m4b-7nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e4m4b_7nih.dat --simulate -v1

- e5m3b-15nih

In [ ]:
%mkdir -p /content/truefloat/e5m3b_15nih
%cd /content/truefloat/e5m3b_15nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e5m3b-15nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e5m3b_15nih.dat --simulate -v1

After the different variants have been generated, let's have a look at the generated output to verify the impact of the different custom floating-point formats on the application results.

In [ ]:
%cd /content/truefloat
from multi_plot import main as plot_waves

from pathlib import Path

base_dir = Path("/content/truefloat")
dat_files = []

for subdir in base_dir.iterdir():
    if subdir.is_dir():
        dat_files.extend(subdir.glob("*.dat"))

plot_waves(dat_files)

---

## <font color='#088F8F'> **Exercise 4: Interface generation** </font>

In this exercise, we explore how different `#pragma` directives in the input code affect the generation of accelerator interfaces, focusing on **AXI4 ports and caches**.

Here are the input files for a simpke kernel and its testbench, containing a directive specifying parameters for the generation of an AXI4 interface:

In [ ]:
%cd /content/Axi4/Exercise1
print("=== Source file: read.c ===\n")
!cat read.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

Now, we run Bambu to generate and simulate the accelerator. We need to add the `--generate-interface=INFER` command-line option to enable parsing of pragma directives referring to top function arguments.


In [ ]:
%cd /content/Axi4/Exercise1
!bambu --generate-interface=INFER --generate-tb=testbench.c read.c --top-fname=read --simulate -v1

Let's look at the ports of the top-level function:


In [ ]:
print_last_verilog_interface('/content/Axi4/Exercise1/read.v')

Next, we will add **multiple `#pragma HLS interface` directives** to a different example to generate **distinct AXI4 ports**, one for each function parameter. This is useful when the accelerator needs to access multiple independent memory regions in parallel.


In [ ]:
%cd /content/Axi4/Exercise2
print("=== Source file: sum.c ===\n")
!cat sum.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

This setup allows Bambu to simulate independent memory accesses for each port:

In [ ]:
%cd /content/Axi4/Exercise2
!bambu --generate-interface=INFER --generate-tb=testbench.c sum.c --top-fname=sum --simulate -v1

Let's look at the ports of the top-level function:

In [ ]:
print_last_verilog_interface('/content/Axi4/Exercise2/sum.v')

How much performance was gained (or lost) by having two separate interfaces for the two arguments? We can use the `bundle` clause to assign both function parameters to the **same AXI4 bundle** (named `test`). This technique is useful for grouping related memory accesses under one interface.


In [ ]:
%cd /content/Axi4/Exercise3
print("=== Source file: sum.c ===\n")
!cat sum.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

Let's see the result:

In [ ]:
%cd /content/Axi4/Exercise3
!bambu --generate-interface=INFER --generate-tb=testbench.c sum.c --top-fname=sum --simulate -v1

And here are the ports of the top-level function:

In [ ]:
print_last_verilog_interface('/content/Axi4/Exercise3/sum.v')

Next, let's apply multiple AXI4 interfaces to a **matrix multiplication** kernel to allow parallel memory accesses to each argument.


In [ ]:
%cd /content/Axi4/Exercise4
print("=== Source file: mmult.c ===\n")
!cat mmult.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

We run Bambu and simulate realistic memory behavior with read and write delays for the external memory using the `--mem-delay-read` and `--mem-delay-write` options.

In [ ]:
%cd /content/Axi4/Exercise4
!bambu --generate-interface=INFER --generate-tb=testbench.c mmult.c --top-fname=mmult --simulate -v1 --mem-delay-read=20 --mem-delay-write=20

Now that we have run Bambu, let's look at the ports of the top-level function:

In [ ]:
print_last_verilog_interface('/content/Axi4/Exercise4/mmult.v')

Finally, let's add **cache pragmas** to the AXI4 bundles.

In [ ]:
%cd /content/Axi4/Exercise5
print("=== Source file: mmult.c ===\n")
!cat mmult.c
print("\n\n=== Testbench file: testbench.c ===\n")
!cat testbench.c

Let's see if the addition of the caches provided improvements in performance:


In [ ]:
%cd /content/Axi4/Exercise5
!bambu --generate-interface=INFER --generate-tb=testbench.c mmult.c --top-fname=mmult --simulate -v1 --mem-delay-read=20 --mem-delay-write=20

Now that we have run Bambu, let's look at the ports of the top-level function by extracting the interface of the top module generated.

In [ ]:
print_last_verilog_interface('/content/Axi4/Exercise5/mmult.v')

Now check if the caches are present:

In [ ]:
import re

def print_verilog_module_interface(filename, module_name="IOB_cache_axi"):
    with open(filename, 'r') as f:
        content = f.read()

    pattern = rf'module\s+{module_name}\s*\(([^;]+);'
    match = re.search(pattern, content, re.DOTALL)

    if not match:
        print(f"Module '{module_name}' not found in {filename}")
        return

    interface = match.group(0)
    print(f"Interface of the `{module_name}` module in {filename}:\n")
    print(interface)

print_verilog_module_interface('/content/Axi4/Exercise5/mmult.v')